In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Functions

### Constants

In [2]:
str_dirname_output = './output'

### Output directory

In [3]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Load Gen 12 v2 test data used for informing the Gen 12 model's class weights

In [4]:
list_cols =  [
    'bigAccountId',
]
str_filename = 'df_test_raw.gzip'
str_uri = f's3://20240327-genxii-v2/04_pd/01_data_prep/03_train_valid_test_split/{str_filename}'
df = pd.read_parquet(str_uri, columns=list_cols)
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:283: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,bigAccountId
12946,5902015
12590,5902037
13218,5902110
14629,5902134
12750,5902141
...,...
16040,5951271
18724,5951279
17722,5951280
15435,5951307


### Load targets

In [5]:
str_filename = 'df_early_indicator_targets.csv'
str_local_path = f'../01_early_indicator_analysis/output/{str_filename}'
list_cols = list(pd.read_csv(str_local_path)['col'])

# get the early indicator targets
list_cols = ['bigAccountId'] + list_cols
str_filename = 'df_targets.gzip'
str_uri = f's3://20240327-genxii-v2/02_target_creation/01_classification/{str_filename}'
df_tmp = pd.read_parquet(str_uri, columns=list_cols)
df_tmp

,bigAccountId,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag
0,6,0,0,0,0
1,25,1,0,0,1
2,73,0,0,0,0
3,82,1,1,1,1
4,122,0,0,0,1
...,...,...,...,...,...
343069,8619039,0,0,0,0
343070,8619083,0,0,0,0
343071,8619360,0,0,0,0
343072,8619910,0,0,0,0


### Join

In [6]:
df = pd.merge(
    left=df,
    right=df_tmp,
    on='bigAccountId',
    how='left',
)
df

,bigAccountId,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag
0,5902015,0,0,0,0
1,5902037,0,0,0,1
2,5902110,0,0,0,0
3,5902134,1,1,1,1
4,5902141,1,1,1,1
...,...,...,...,...,...
3222,5951271,0,0,0,1
3223,5951279,0,0,0,0
3224,5951280,0,0,0,1
3225,5951307,0,0,1,1


### Get the mean of all targets

In [7]:
list_cols = [col for col in df.columns if col != 'bigAccountId']
ser_mean = df[list_cols].mean()
df_mean = ser_mean.reset_index()
df_mean.columns = ['flag','mean']
# show
df_mean

,flag,mean
0,Early_Pay_Delinquency_15_60_Flag,0.141308
1,Early_Pay_Delinquency_30_90_Flag,0.127363
2,Early_Pay_Delinquency_30_180_Flag,0.305237
3,Early_Pay_Delinquency_30_360_Flag,0.522467


### Save

In [8]:
str_filename = 'df_mean.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_mean.to_csv(str_local_path, index=False)